## pull data from API 

requests -> get 

In [0]:
import requests
import json
from datetime import datetime

url = "https://inceptezlabs.com/api.php"

headers = { "User-Agent": "Chrome/151.0.0.0 Safari/537.36" }        
resp = requests.get(url, headers=headers)
data=resp.json()
ts = datetime.now().strftime("%Y%m%d%H%M%S")
output_path = f"/Volumes/lakehousecat1/deltadb/datalake/wd37src/apidata/posts_{ts}.json"

dbutils.fs.put( output_path,
                json.dumps(data),
                overwrite=True
            )


## ingest data from cloud storage to Lakehouse 

In [0]:
df_raw = (spark.readStream
        .format("cloudFiles")#.schema(user_schema)
        .option("cloudFiles.format", "json")
        #.option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.inferColumnTypes",True)\
        .option("cloudFiles.schemaLocation","/Volumes/lakehousecat1/deltadb/datalake/wd37src/apischema2/")
        .option("cloudFiles.maxFilesPerTrigger", 1)
        .load("/Volumes/lakehousecat1/deltadb/datalake/wd37src/apidata/"))

In [0]:

df_select=df_raw.select("data.uid","data.user.name","data.user.email","data.user.location")

(df_select.writeStream
        .format("delta")
        .trigger(availableNow=True)
        .option(
            "checkpointLocation",
            "/Volumes/lakehousecat1/deltadb/datalake/wd37src/apicheckpoint")
        .toTable("lakehousecat1.deltadb.apidatajson_simplified"))

In [0]:
%sql

select * from lakehousecat1.deltadb.apidatajson_simplified

In [0]:
%sql
-- complex types in DBSQL / Hive 
-- array , map , struct 
-- array - ordered collection of similar elements , index
-- map - un ordered collection of key value pair , k,v 
-- struct - collection of named fields , - field name

--drop table izwd37dev.wd37db.userdetail;

-- array --> list
-- map ---> dict
--create catalog if not exists izwd37dev;
create or replace table izwd37dev.wd37db.userdetail
(
    id int ,
    name string,
    subjects array<string>, 
    address struct<street:string,city:string,pin:int>,
    marks map<string,int>
);

insert into izwd37dev.wd37db.userdetail values(1,'John',array('python','sql'),struct('main street','NY',12345),map('maths',90,'science',80));
insert into izwd37dev.wd37db.userdetail values(2,'Mary',array('scala','sql'),struct('10th street','LA',23456),map('maths',92,'science',88));
insert into izwd37dev.wd37db.userdetail values(3,'Ravi',array('java','sql'),struct('main street','NY',12345),map('maths',70,'science',60));

select * from izwd37dev.wd37db.userdetail ;

describe formatted izwd37dev.wd37db.userdetail;

-- id,name,fav_sub,city,pincode,maths_mark

select id,name,subjects[0] || ' ' || subjects[1]  as fav_sub,address.street,address.city ,address.pin,marks['maths'] as maths_mark from izwd37dev.wd37db.userdetail ;

